# 2주차 실습 (05) - 미니 autograd 엔진 (micrograd 스타일)

값을 감싸 연산 그래프를 기록하고, 위상정렬 후 역순으로 연쇄 법칙을 적용해
**reverse-mode 자동미분**을 구현한다. 이걸로 XOR MLP까지 학습한다.

## Step 1. `Value` 클래스 — 값 + grad + 연산 기록

- 모든 연산이 `_backward` 클로저(국소 그래디언트를 부모로 전파)를 만든다.
- 한 값이 여러 연산에 쓰이면 `grad += ...` (gradient accumulation).

In [ ]:
import math


class Value:
    def __init__(self, data, children=(), op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad          # d(a+b)/da = 1
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f"**{n}")

        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def _backward():
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    # 파생 연산 (기존 연산으로 정의 -> 그래디언트 공짜)
    def __neg__(self):        return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def backward(self):
        # 위상정렬: 모든 의존 노드가 뒤에 오도록
        topo, visited = [], set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1.0                    # 시드 dy/dy = 1
        for v in reversed(topo):
            v._backward()

## Step 2. 수동 검증: `y = relu(x1*x2 + 1)`

`x1*x2+1 = 7 > 0` 이라 relu는 항등 → `dy/dx1 = x2 = 3`, `dy/dx2 = x1 = 2`.

In [ ]:
x1, x2 = Value(2.0), Value(3.0)
y = (x1 * x2 + 1).relu()
y.backward()
print(f"y = {y.data}  dy/dx1 = {x1.grad} (=x2)  dy/dx2 = {x2.grad} (=x1)")

## Step 3. 그래디언트 체킹

autodiff 결과를 수치 미분 `(f(x+h)-f(x-h))/2h` 과 비교. 차이 < 1e-5 면 OK.

In [ ]:
def gradient_check(build_expr, x_val, h=1e-7):
    x = Value(x_val)
    y = build_expr(x)
    y.backward()
    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical = (y_plus - y_minus) / (2 * h)
    return x.grad, numerical, abs(x.grad - numerical)


ad, num, diff = gradient_check(lambda x: (x ** 3 + x * 2 + 1).tanh(), 0.5)
print(f"autodiff={ad:.8f}  numerical={num:.8f}  diff={diff:.2e}")

## Step 4. 미니 MLP

`Neuron`(가중합+tanh) → `Layer`(뉴런 리스트) → `MLP`(층 스택). 모든 가중치가 `Value`라 `loss.backward()`가 전부에 그래디언트를 전파.

In [ ]:
import random


class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Step 5. XOR 학습

선형 분리 불가능한 XOR을 미니 autograd 엔진만으로 학습 (tanh 출력이라 타깃 -1/1).

In [ ]:
random.seed(42)
model = MLP([2, 4, 1])
xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [-1, 1, 1, -1]

for step in range(100):
    preds = [model(x) for x in xs]
    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    for p in model.parameters():
        p.data -= 0.05 * p.grad          # 경사하강
    if step % 20 == 0:
        print(f"step {step:3d}  loss = {loss.data:.4f}")

print("\n학습 후 예측:")
for x, y in zip(xs, ys):
    print(f"  input={x}  target={y:2d}  pred={model(x).data:6.3f}")

## 정리

- 값 래핑 + 연산 기록 + 위상정렬 backward = PyTorch `autograd`의 축소판.
- 신경망은 입력(가중치) 다 · 출력(손실) 1 → **reverse mode** 한 패스로 모든 그래디언트.
- 개념 노트: [[🔁 자동미분과 역전파]] (Obsidian LLM_Wiki)